In [158]:
import pandas as pd
import numpy as np
import json

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.ticker import FuncFormatter
import plotly.express as px

import math
from collections import Counter, defaultdict
import re
from unidecode import unidecode



In [159]:
df = pd.read_csv("../../data/papers.csv")
total_papers = len(df)
df.columns

Index(['Title', 'Database', 'Year', 'Month', 'Journal', 'Paper Type',
       'Data Source', 'Data Country', 'Data Domain', 'Data Language',
       'Data Availability', 'Link to Data', 'Data Details', 'Size',
       'Number of Prompts', 'Medical Application', 'Task Type', 'Topic',
       'Note', 'Bias Evaluation Metric', 'Bias Definition', 'Conclusions',
       'Race / Ethnic Bias', 'Language Bias', 'Age Bias', 'Gender Bias',
       'Other Bias', 'LGBTQ+ Bias', 'Disability Bias',
       'Geography / Cultural Bias', 'Evaluated LLMs',
       'Reference Standard \r\n(Human / Model / System / Physician)',
       'Patient Inclusion\r\n(Yes / No)', 'Has Debiasing',
       'Debias - Focus\r\n(Data/Train/Inference)', 'Debias-details'],
      dtype='str')

1. get a language to country map
2. get language to iso_alpha
3. convert to dataframe

formatted like this ...

map = pd.DataFrame({
    "country": ["United States", "France", "China", "Brazil", "India"],
    "iso_alpha": ["USA", "FRA", "CHN", "BRA", "IND"],
    "frequency": [78.2, 82.5, 77.1, 75.5, 69.7]
})

In [160]:
with open('../languages_to_countries/languages_to_countries.json', 'r', encoding='utf-8') as file:
    language_to_countries = json.load(file)

country_to_iso_alpha = pd.read_csv("../languages_to_countries/data/iso.csv")
country_to_iso_alpha = country_to_iso_alpha[["name", "alpha-3"]]

# languages_to_countries.json uses shorter/common names; iso.csv uses ISO official names.
ISO_NAME_ALIASES = {
    "United States": "United States of America",
    "U.S. Minor Outlying Islands": "United States Minor Outlying Islands",
    "Cocos [Keeling] Islands": "Cocos (Keeling) Islands",
    "Falkland Islands": "Falkland Islands (Malvinas)",
    "Micronesia": "Micronesia, Federated States of",
    "Swaziland": "Eswatini",
    "Ivory Coast": "Côte d'Ivoire",
    "Democratic Republic of the Congo": "Congo, Democratic Republic of the",
    "Republic of the Congo": "Congo",
    "Bolivia": "Bolivia, Plurinational State of",
    "Venezuela": "Venezuela, Bolivarian Republic of",
    "Russia": "Russian Federation",
    "Czech Republic": "Czechia",
    "Turkey": "Türkiye",
    "Palestine": "Palestine, State of",
    "East Timor": "Timor-Leste",
    "Pitcairn Islands": "Pitcairn",
    "Cape Verde": "Cabo Verde",
}

In [161]:
language_groups_lst = df["Language Bias"].tolist()
language_groups_freqs = defaultdict(int)

for s in language_groups_lst:
    
    if isinstance(s, float):
        language_groups_freqs[str(s)] += 1
        continue
    
    s = s.strip().lower()
    s = s.replace("_", " ")
    
    if s == "no":
        language_groups_freqs[s] += 1
        continue
    
    match = re.search(r'\((.*?)\)', s)
    if not match:
        continue   # skip malformed rows safely
    
    items = match.group(1)   # <-- only inside parentheses
    languages = [lang.strip() for lang in items.split(',')]
    
    for language in languages:
        language_groups_freqs[language] += 1

# Cleaning... subject to change
del language_groups_freqs["no"]

language_groups_freqs["english"] += language_groups_freqs["aave"]
del language_groups_freqs["aave"]
del language_groups_freqs["standard english"]

language_groups_freqs["arabic"] += 1
del language_groups_freqs["saudi arabic"]
del language_groups_freqs["egyptian arabic"]
del language_groups_freqs["lebanese arabic"]
del language_groups_freqs["moroccan arabic"]
del language_groups_freqs["non-arabic"]

language_groups_freqs["english"] += 1
del language_groups_freqs["traditional chinese"]
del language_groups_freqs["simplified chinese"]

language_groups_freqs["portuguese"] = 1
    

In [162]:
language_groups_freqs

defaultdict(int,
            {'chinese': 2,
             'english': 12,
             'arabic': 3,
             'spanish': 4,
             'polish': 1,
             'kazakh': 1,
             'russian': 1,
             'japanese': 1,
             'dutch': 1,
             'portuguese': 1})

In [163]:
country_to_freqs = defaultdict(int)

for language, frequency in language_groups_freqs.items():
    countries_who_speak_it = language_to_countries[language.capitalize()]
    for country in countries_who_speak_it:
        country = unidecode(country)
        country_to_freqs[country] += frequency



In [164]:
country_to_freqs

defaultdict(int,
            {'China': 2,
             'Hong Kong': 14,
             'Macao': 3,
             'Singapore': 14,
             'Taiwan': 2,
             'United States': 12,
             'Canada': 12,
             'American Samoa': 12,
             'Australia': 12,
             'Botswana': 12,
             'Cocos [Keeling] Islands': 12,
             'Cook Islands': 12,
             'Cameroon': 12,
             'Christmas Island': 12,
             'Eritrea': 15,
             'Fiji': 12,
             'Falkland Islands': 12,
             'Micronesia': 12,
             'United Kingdom': 12,
             'Guernsey': 12,
             'Ghana': 12,
             'Gibraltar': 12,
             'Gambia': 12,
             'South Georgia and the South Sandwich Islands': 12,
             'Guam': 16,
             'Guyana': 12,
             'Heard Island and McDonald Islands': 12,
             'Ireland': 12,
             'Isle of Man': 12,
             'India': 12,
             'British In

In [165]:
countries = []
iso_alphas = []
frequencies = []

iso_lookup = country_to_iso_alpha.set_index("name")["alpha-3"]

for country, freq in country_to_freqs.items():
    countries.append(country)

    iso_name = ISO_NAME_ALIASES.get(country, country)
    alpha = iso_lookup.get(iso_name)
    iso_alphas.append(alpha)

    frequencies.append(freq)


In [166]:
map = pd.DataFrame({
    "country": countries,
    "iso_alpha": iso_alphas,
    "frequency": frequencies
})

In [167]:
import plotly.io as pio
pio.renderers.default = "browser"

fig = px.choropleth(
    map,
    locations="iso_alpha",
    color="frequency",
    hover_name="country",
    color_continuous_scale=px.colors.sequential.Plasma,
    labels={
        "frequency": "No. of Papers",
    },
)

fig.update_layout(
    title=dict(
        text="Number of Papers Studying Languages of that Country",
        font=dict(size=24),
        x=0.5,
        xanchor="center",
    ),
)

fig.show()